# 04 · Analysis, tables and figures

**Settings:** Accelerator *None*, Internet on (only for the circuit drawing; everything else works offline).
**Inputs:** `bbd-code`, the output of notebook 01, the output of the **latest** version of notebook 02 (it already
contains all earlier versions' results), and the output of notebook 03 if the hardware run is done.

Produces every figure (PDF + 300 dpi PNG, drawn at Springer LNCS size), the LaTeX tables and the
pre-registered hypothesis tests, and zips them as `paper_assets.zip`. It can be run at any point of the grid:
figures whose runs are missing are skipped.

In [ ]:
# Pinned: these are the versions the hardware code was tested with (SamplerV2 API).
%pip install -q "qiskit==2.6.1" "qiskit-ibm-runtime==0.51.0" "qiskit-aer==0.18.0" pylatexenc

In [ ]:
# --- bootstrap: find the bbd-code dataset, copy it to a writable folder, import it
import glob, os, shutil, sys
# only the bbd-code dataset counts: attached notebook outputs never contain the code (it is copied to /tmp)
hits = [h for h in sorted(glob.glob("/kaggle/input/**/bbd/__init__.py", recursive=True)) if "/bbd_code/" not in h]
hits = hits or sorted(glob.glob("./bbd/__init__.py"))
if not hits:
    raise FileNotFoundError("Attach the 'bbd-code' dataset to this notebook (KAGGLE_INSTRUCTIONS.md, step 2).")
if len(hits) > 1:
    print("warning: several copies of the code are attached; using", hits[0])
SRC = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
CODE = "/tmp/bbd_code" if os.path.isdir("/kaggle/working") else SRC
if CODE != SRC:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "results", "work"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
import bbd
print("bbd", bbd.__version__, "loaded from", SRC)

In [ ]:
import json, shutil
from pathlib import Path
from bbd import config, figures, runner, stats, tables
RESULTS = config.RESULTS_DIR
runner.carry_forward(RESULTS, config.INPUT_ROOT, kinds=("ledger",))   # artifacts are read from the inputs in place
hw_out = None
for f in sorted(Path(config.INPUT_ROOT).glob("**/hardware/hw_results.json")):
    hw_out = json.loads(f.read_text()); print("hardware results:", f)
print(json.dumps(runner.status(RESULTS), indent=1))
df = stats.records_frame(RESULTS)
print(len(df), "records,", df["run_id"].nunique(), "runs")

## Pre-registered hypotheses (H1–H3) and the saturation regression

In [ ]:
hyp = stats.hypothesis_table(df)
display(hyp.reindex(columns=["hypothesis", "role", "claim", "test", "n", "est", "ci90", "pi95", "p", "p_holm"]))
print("saturation regression (tail-recall gain vs saturation):", stats.saturation_regression(df))

In [ ]:
tables.write_all(df, hyp, config.TABLE_DIR)
print(sorted(p.name for p in config.TABLE_DIR.iterdir()))
print((config.TABLE_DIR / "table_main_toniot8.tex").read_text()[:1500])

## Figures

In [ ]:
shot_df = figures.shot_curve(RESULTS, df)        # CPU simulation from saved checkpoints
made = figures.make_all(df, config.FIG_DIR, RESULTS, hw_out, shot_df)
try:
    from bbd.hardware import draw_circuit
    draw_circuit((1, 10, 2), 9, config.FIG_DIR); print("ok   fig0_circuit")
except Exception as e:
    print("skip fig0_circuit:", e)
from bbd import data
try:
    figures.fig_datasets([data.load_prepared("toniot8"), data.load_prepared("ciciot8")], config.FIG_DIR)
except Exception as e:
    print("skip fig_datasets:", e)

In [ ]:
from IPython.display import Image, display
for p in sorted(config.FIG_DIR.glob("*.png")):
    print(p.name); display(Image(str(p), width=860))

In [ ]:
import zipfile
ZIP = config.WORK_ROOT / "paper_assets.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (config.FIG_DIR, config.TABLE_DIR):
        for p in folder.iterdir():
            z.write(p, f"{folder.name}/{p.name}")
print("wrote", ZIP)